# Wikidata Candidate Sense Merge Experiment

This notebook tests whether an OpenAI API LLM can merge Wikidata candidate senses that are semantically close or duplicated. It uses the same candidate-loading utility used by `LiteSemRAG` semantic-description assignment, then asks the LLM to produce a smaller cleaned candidate set.

## Parameters

Change `WORD` and `MAX_CANDIDATE_COUNT`, then run the notebook top to bottom. The OpenAI key is read from `OPENAI_API_KEY` if present, otherwise from the root `API_KEY` file.

In [1]:
from pathlib import Path
import json
import os
import pickle
import re
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModel, AutoTokenizer

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from text_processing import get_token_indices_for_phrase, normalize_text
from utils import (
    build_wikidata_candidate_bank,
    farthest_first_traversal,
    load_wikidata_definition_candidates,
)

MAX_CANDIDATE_COUNT = 8
USE_DETAILED_DESCRIPTION = False
USE_WIKIDATA_SPAN_RULES = False

HOTPOT_SCAN_STORE_PATH = REPO_ROOT / "data/hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl"
FFT_SAMPLE_COUNT = 15
INPUT_SAMPLE_LIMIT = 150
PROMPT_CONTEXT_MODE = "sentence_neighbors"
TEXT_ENCODER_NAME_OR_PATH = "/home/xiaoyue/ProtoGraphRAG/deberta-v3-large"
EMBED_BATCH_SIZE = 16
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OPENAI_MODEL = "gpt-5.4-mini"
API_KEY_PATH = REPO_ROOT / "API_KEY"


def load_local_api_config(path: Path = API_KEY_PATH) -> dict:
    if not path.exists():
        return {}
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return {}
    if text.startswith("{"):
        return json.loads(text)
    if "=" not in text:
        return {"OPENAI_API_KEY": clean_api_value(text)}
    config = {}
    for part in text.replace(";", "\n").splitlines():
        part = part.strip()
        if not part or part.startswith("#") or "=" not in part:
            continue
        key, value = part.split("=", 1)
        config[key.strip().strip("'\"")] = clean_api_value(value)
    return config


def clean_api_value(value: str) -> str:
    value = str(value).strip()
    value = value.removesuffix(",").strip()
    value = value.strip("'\"")
    value = value.replace("\\n", "").replace("\\r", "").strip()
    value = value.removesuffix(",").strip()
    value = value.strip("'\"")
    return value


def first_config_value(config: dict, *keys: str):
    for key in keys:
        value = config.get(key)
        if value:
            return value
    return None


LOCAL_API_CONFIG = load_local_api_config()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or first_config_value(
    LOCAL_API_CONFIG,
    "OPENAI_API_KEY",
    "openai_api_key",
    "api_key",
    "chatgpt_api",
)
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL") or first_config_value(
    LOCAL_API_CONFIG,
    "OPENAI_BASE_URL",
    "openai_base_url",
    "base_url",
)

print(f"repo: {REPO_ROOT}")
print(f"local API key file exists: {API_KEY_PATH.exists()}")
print(f"OpenAI key loaded: {bool(OPENAI_API_KEY)}")
print(f"OpenAI base URL: {OPENAI_BASE_URL or '(default)'}")

repo: /home/xiaoyue/LiteSemRAG
local API key file exists: True
OpenAI key loaded: True
OpenAI base URL: (default)


## 1. Fetch Wikidata Candidate Senses

`MAX_CANDIDATE_COUNT` is passed both as the Wikidata search limit and the post-filter target count, so this cell requests up to that many usable definitions. Set `USE_WIKIDATA_SPAN_RULES = False` to use raw Wikidata candidates without the span-aware label/alias rules.

In [2]:
def fetch_candidate_senses(
    word: str,
    max_candidates: int,
    use_detailed_description: bool = False,
    use_span_rules: bool = True,
):
    candidates_df, definition_column = load_wikidata_definition_candidates(
        word,
        use_detailed_description=use_detailed_description,
        limit=max_candidates,
        target_candidate_count=max_candidates,
        use_span_rules=use_span_rules,
    )
    candidate_bank = build_wikidata_candidate_bank(
        candidates_df,
        definition_column=definition_column,
    )
    rows = []
    for idx, candidate in enumerate(candidate_bank, start=1):
        rows.append(
            {
                "candidate_id": idx,
                "wikidata_id": candidate["entity_id"],
                "label": candidate["label"],
                "description": candidate["description"],
                "definition": candidate["definition"],
                "hypothesis": candidate["hypothesis"],
            }
        )
    return rows, candidates_df

WORD = "space"
candidate_senses, raw_candidates_df = fetch_candidate_senses(
    WORD,
    MAX_CANDIDATE_COUNT,
    use_detailed_description=USE_DETAILED_DESCRIPTION,
    use_span_rules=False,
)

candidate_df = pd.DataFrame(candidate_senses)
candidate_df

,candidate_id,wikidata_id,label,description,definition,hypothesis
0,1,Q1,universe,"totality consisting of space, time, matter and...","totality consisting of space, time, matter and...","It refers to totality consisting of space, tim..."
1,2,Q380933,space,"blank area that separates words, sentences, sy...","blank area that separates words, sentences, sy...","It refers to blank area that separates words, ..."
2,3,Q4169,outer space,void between celestial bodies,void between celestial bodies,It refers to void between celestial bodies.
3,4,Q61977583,Space,2014 video game,2014 video game,It refers to 2014 video game.
4,5,Q63968426,S P A C E,"hackspace in Nuremberg, Germany","hackspace in Nuremberg, Germany","It refers to hackspace in Nuremberg, Germany."
5,6,Q107,space,three-dimensional extent in which objects exis...,three-dimensional extent in which objects exis...,It refers to three-dimensional extent in which...
6,7,Q193701,SpaceX,American private aerospace company,American private aerospace company,It refers to American private aerospace company.
7,8,Q16555,Houston,"seat of Harris County, and largest city in Sta...","seat of Harris County, and largest city in Sta...","It refers to seat of Harris County, and larges..."


## 2. Select FFT Dataset Examples

Use the HotpotQA scan store and the same span-embedding plus `farthest_first_traversal(..., start="random")` sampling pattern as `hotpotqa_fft_span_compare.ipynb`. These examples are collected first and are not inserted into the LLM merge prompt yet.

In [3]:
text_tokenizer = None
text_encoder_model = None


def load_hotpot_scan_store(path: Path = HOTPOT_SCAN_STORE_PATH):
    if not path.exists():
        raise FileNotFoundError(f"HotpotQA scan store not found: {path}")
    with path.open("rb") as handle:
        return pickle.load(handle)


def lookup_records(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def _find_left_boundary(text: str, index: int) -> int:
    return max(
        text.rfind(".", 0, index),
        text.rfind("!", 0, index),
        text.rfind("?", 0, index),
    )


def _find_right_boundary(text: str, index: int) -> int:
    right_candidates = [
        text.find(".", index),
        text.find("!", index),
        text.find("?", index),
    ]
    right_candidates = [idx for idx in right_candidates if idx != -1]
    return len(text) if not right_candidates else min(right_candidates) + 1


def _build_context_from_bounds(cleaned_text, span, context_start, context_end):
    start_char, end_char = span
    context_raw = cleaned_text[context_start:context_end]

    if not context_raw.strip():
        context_start = max(0, start_char - 120)
        context_end = min(len(cleaned_text), end_char + 120)
        context_raw = cleaned_text[context_start:context_end]

    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - context_start - left_trim
    local_end = end_char - context_start - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def extract_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)
    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_neighbor_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)

    if context_start > 0:
        previous_boundary = _find_left_boundary(cleaned_text, max(0, context_start - 1))
        context_start = 0 if previous_boundary == -1 else previous_boundary + 1

    if context_end < len(cleaned_text):
        context_end = _find_right_boundary(cleaned_text, context_end)

    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_full_context(cleaned_text, span):
    start_char, end_char = span
    context_raw = cleaned_text
    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - left_trim
    local_end = end_char - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def extract_prompt_context(cleaned_text, span, prompt_context_mode="sentence"):
    if prompt_context_mode == "sentence":
        return extract_sentence_context(cleaned_text, span)
    if prompt_context_mode == "full_text":
        return extract_full_context(cleaned_text, span)
    if prompt_context_mode == "sentence_neighbors":
        return extract_neighbor_sentence_context(cleaned_text, span)
    raise ValueError(
        f"Unsupported prompt_context_mode={prompt_context_mode!r}. Use 'sentence', 'sentence_neighbors', or 'full_text'."
    )


def build_hotpot_prompt(
    record,
    query_text,
    mark_target=False,
    left_marker="[TGT]",
    right_marker="[/TGT]",
    prompt_context_mode="sentence",
):
    context_info = extract_prompt_context(
        record["cleaned_text"],
        record["span"],
        prompt_context_mode=prompt_context_mode,
    )
    context_text = context_info["context_text"]
    local_start, local_end = context_info["local_span"]
    prompt_context = context_text

    if mark_target:
        prompt_context = (
            f"{context_text[:local_start]}{left_marker} {context_text[local_start:local_end]} {right_marker}{context_text[local_end:]}"
        )

    prompt_text = (
        f"Context: {prompt_context}\n"
        f"Target word: {query_text}\n"
        "Which Wikidata candidate best matches the target word in this context?"
    )
    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
        "matched_text": context_text[local_start:local_end],
        "prompt_text": prompt_text,
    }


def load_text_encoder(name_or_path: str, device: str):
    loaded_tokenizer = AutoTokenizer.from_pretrained(
        name_or_path,
        local_files_only=True,
        fix_mistral_regex=True,
        use_fast=True,
    )
    if not getattr(loaded_tokenizer, "is_fast", False):
        raise RuntimeError(
            "The text encoder tokenizer must be a fast tokenizer because offset_mapping is required."
        )

    loaded_model = AutoModel.from_pretrained(
        name_or_path,
        local_files_only=True,
    )
    loaded_model.to(device)
    loaded_model.eval()
    return loaded_tokenizer, loaded_model


def ensure_text_encoder_loaded(name_or_path: str, device: str):
    global text_tokenizer, text_encoder_model
    if text_tokenizer is None or text_encoder_model is None:
        text_tokenizer, text_encoder_model = load_text_encoder(name_or_path, device)
        print(f"Loaded text encoder on {device}: {name_or_path}")
    return text_tokenizer, text_encoder_model


def encode_span_text_batch(text_list, tokenizer, model, device, max_length=512):
    encoded_inputs = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        return_tensors="pt",
    )
    offsets = encoded_inputs["offset_mapping"]
    model_inputs = {key: value.to(device) for key, value in encoded_inputs.items() if key != "offset_mapping"}

    with torch.no_grad():
        outputs = model(**model_inputs, output_hidden_states=True)
        token_embeddings = outputs.hidden_states[-2].detach().cpu()

    return token_embeddings, offsets


def collect_query_span_embeddings(
    store,
    query_text,
    kind=None,
    batch_size=EMBED_BATCH_SIZE,
    max_length=MAX_LENGTH,
    prompt_context_mode=PROMPT_CONTEXT_MODE,
    mark_target=False,
    max_records=None,
    text_encoder_name_or_path=TEXT_ENCODER_NAME_OR_PATH,
    device=DEVICE,
):
    normalized_query = normalize_text(query_text.strip())
    query_records = lookup_records(
        store,
        query_text,
        kind=kind,
        include_text=True,
        include_cleaned_text=True,
    )

    if not query_records:
        raise ValueError(f"No records found for span={query_text!r}.")

    if max_records is not None:
        query_records = query_records[: int(max_records)]

    embedding_cache = store.setdefault("embedding_cache", {})
    tokenizer, model = ensure_text_encoder_loaded(text_encoder_name_or_path, device)

    total_records = len(query_records)
    progress_handle = display(
        f"Embedded 0/{total_records} texts for query {normalized_query!r}",
        display_id=True,
    )

    embedded_records = []
    cache_hits = 0
    cache_misses = 0

    for batch_start in range(0, total_records, batch_size):
        batch_end = min(batch_start + batch_size, total_records)
        batch_records = query_records[batch_start:batch_end]

        context_texts = []
        local_spans = []
        matched_texts = []
        batch_embeddings = [None] * len(batch_records)
        uncached_texts = []
        uncached_indices = []
        cache_keys = []

        for local_idx, record in enumerate(batch_records):
            prompt_info = build_hotpot_prompt(
                record,
                query_text,
                mark_target=mark_target,
                prompt_context_mode=prompt_context_mode,
            )
            cache_key = (
                int(record["document_idx"]),
                tuple(record["span"]),
                prompt_context_mode,
                prompt_info["context_text"],
                tuple(prompt_info["local_span"]),
                int(max_length),
                text_encoder_name_or_path,
                "litsemrag_hidden_states_minus_2_mean_pool_context_only",
            )

            matched_texts.append(prompt_info["matched_text"])
            context_texts.append(prompt_info["context_text"])
            local_spans.append(prompt_info["local_span"])
            cache_keys.append(cache_key)

            cached_embedding = embedding_cache.get(cache_key)
            if cached_embedding is None:
                uncached_texts.append(prompt_info["context_text"])
                uncached_indices.append(local_idx)
                cache_misses += 1
            else:
                batch_embeddings[local_idx] = cached_embedding
                cache_hits += 1

        if uncached_texts:
            token_embeddings_batch, offsets_batch = encode_span_text_batch(
                uncached_texts,
                tokenizer,
                model,
                device,
                max_length=max_length,
            )
            for local_idx, token_embeddings, offsets in zip(
                uncached_indices,
                token_embeddings_batch,
                offsets_batch,
            ):
                start_char, end_char = local_spans[local_idx]
                token_indices = get_token_indices_for_phrase(start_char, end_char, offsets.tolist())
                if not token_indices:
                    raise ValueError(
                        f"No tokenizer offsets were found for local_span={(start_char, end_char)} in title={batch_records[local_idx]['title']!r}."
                    )
                embedding = token_embeddings[token_indices].mean(dim=0).to(torch.float32).numpy()
                cache_key = cache_keys[local_idx]
                embedding_cache[cache_key] = embedding
                batch_embeddings[local_idx] = embedding

        for record, matched_text, context_text, local_span, embedding in zip(
            batch_records,
            matched_texts,
            context_texts,
            local_spans,
            batch_embeddings,
        ):
            prompt_info = build_hotpot_prompt(
                record,
                query_text,
                mark_target=mark_target,
                prompt_context_mode=prompt_context_mode,
            )
            item = dict(record)
            item["matched_text"] = matched_text
            item["context_text"] = context_text
            item["local_span"] = local_span
            item["prompt_text"] = prompt_info["prompt_text"]
            item["embedding"] = np.asarray(embedding, dtype=np.float32)
            item["record_index"] = len(embedded_records)
            embedded_records.append(item)

        progress_handle.update(
            f"Embedded {batch_end}/{total_records} texts for query {normalized_query!r}"
        )

    return {
        "query_text": query_text,
        "normalized_query": normalized_query,
        "kind": kind,
        "record_count": len(embedded_records),
        "batch_size": batch_size,
        "max_length": max_length,
        "prompt_context_mode": prompt_context_mode,
        "mark_target": mark_target,
        "max_records": max_records,
        "device": device,
        "text_encoder_name_or_path": text_encoder_name_or_path,
        "embedding_method": "LiteSemRAG span mean-pool from hidden_states[-2]",
        "cache_hits": cache_hits,
        "cache_misses": cache_misses,
        "cache_size": len(embedding_cache),
        "records": embedded_records,
    }


def select_fft_sample_records(records, n=FFT_SAMPLE_COUNT):
    if not records:
        raise ValueError("records must be non-empty")
    embeddings = np.stack([record["embedding"] for record in records]).astype(np.float32)
    if len(records) > n:
        sampled_indices = list(farthest_first_traversal(embeddings, n, start="random"))
    else:
        sampled_indices = list(range(len(records)))
    return [records[idx] | {"fft_sample_order": order, "fft_source_position": idx} for order, idx in enumerate(sampled_indices, start=1)]


def build_fft_sample_frame(sample_records):
    rows = []
    for item in sample_records:
        context_text = re.sub(r"\s+", " ", item["context_text"]).strip()
        rows.append(
            {
                "sample_order": item["fft_sample_order"],
                "record_index": item["record_index"],
                "title": item["title"],
                "kind": item["kind"],
                "matched_text": item["matched_text"],
                "local_span": item["local_span"],
                "context_text": context_text,
            }
        )
    return pd.DataFrame(rows)


embedding_store = load_hotpot_scan_store(HOTPOT_SCAN_STORE_PATH)
embedded_result = collect_query_span_embeddings(
    embedding_store,
    WORD,
    batch_size=EMBED_BATCH_SIZE,
    max_length=MAX_LENGTH,
    prompt_context_mode=PROMPT_CONTEXT_MODE,
    max_records=INPUT_SAMPLE_LIMIT,
    text_encoder_name_or_path=TEXT_ENCODER_NAME_OR_PATH,
    device=DEVICE,
)
fft_sample_records = select_fft_sample_records(embedded_result["records"], n=FFT_SAMPLE_COUNT)
fft_sample_df = build_fft_sample_frame(fft_sample_records)

print(f"FFT sample count: {len(fft_sample_records)} / records: {embedded_result['record_count']}")
print(f"Text encoder: {embedded_result['text_encoder_name_or_path']}")
print(f"Embedding method: {embedded_result['embedding_method']}")
display(fft_sample_df)

print("\nFull FFT sample contexts:")
for sample in fft_sample_records:
    context_text = re.sub(r"\s+", " ", sample["context_text"]).strip()
    print("=" * 100)
    print(f"sample_order: {sample['fft_sample_order']}")
    print(f"title: {sample['title']}")
    print(f"matched_text: {sample['matched_text']}")
    print("context_text:")
    print(context_text)


/home/xiaoyue/anaconda3/envs/llm_graph/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Loaded text encoder on cuda: /home/xiaoyue/ProtoGraphRAG/deberta-v3-large


"Embedded 78/78 texts for query 'space'"

FFT sample count: 15 / records: 78
Text encoder: /home/xiaoyue/ProtoGraphRAG/deberta-v3-large
Embedding method: LiteSemRAG span mean-pool from hidden_states[-2]


,sample_order,record_index,title,kind,matched_text,local_span,context_text
0,1,33,Cosmological interpretation of quantum mechanics,token,space,"(608, 613)",the cosmological interpretation of quantum mec...
1,2,29,Enrique Banchs,token,space,"(85, 90)",Enrique Banchs (1888 – 1968) was an argentine ...
2,3,58,HCL color space,token,spaces,"(154, 160)",HCL is designed to have characteristics of bot...
3,4,72,Sam Eagle,token,space,"(258, 263)",Sam Eagle is a muppet character originating fr...
4,5,48,Color space,token,space,"(152, 157)","in combination with physical device profiling,..."
5,6,62,RGB color space,token,space,"(13, 18)",an RGB color space is any additive color space...
6,7,57,HCL color space,token,space,"(129, 134)",HCL is designed to have characteristics of bot...
7,8,11,"Summerlin, Nevada",token,space,"(293, 298)",it lies at the edge of the Spring Mountains an...
8,9,9,Battle Beyond the Sun,token,space,"(125, 130)","version of Nebo Zovyot, a 1959 soviet science ..."
9,10,2,Coral Gardens and Their Magic,token,spaces,"(206, 212)",it concentrates on the cultivation practices t...



Full FFT sample contexts:
sample_order: 1
title: Cosmological interpretation of quantum mechanics
matched_text: space
context_text:
the cosmological interpretation of quantum mechanics, proposed by Anthony Aguirre and Max Tegmark, is an interpretation of quantum mechanics that applies in the context of eternal cosmological inflation, which arguably predicts an infinite three-dimensional space with infinitely many planets and infinitely many copies of any quantum system. according to this interpretation, the wavefunction for a quantum system describes not some imaginary ensemble of possibilities for what the system might be doing, but rather the actual spatial collection of identical copies of the system that exist in our infinite space. its collapse can be avoided.
sample_order: 2
title: Enrique Banchs
matched_text: space
context_text:
Enrique Banchs (1888 – 1968) was an argentine poet. he published all his work in the space of four years at the beginning of the 20th century, then lay

## 3. Prompt The LLM To Merge Similar Senses

The prompt now biases the model toward coarse retrieval-oriented senses: merge institution/role variants that a cross-encoder would struggle to separate, and split only when the meanings would clearly retrieve different facts.

In [4]:
SYSTEM_PROMPT = """You are a conservative lexical-sense merger for a semantic retrieval system.
Your job is to reduce noisy Wikidata candidate senses to a small set of coarse retrieval meanings.
Prefer merging over splitting when candidates describe the same broad concept, role, entity type, or function.
Do not preserve fine-grained domain, institution, jurisdiction, title, or wording differences unless they change what evidence should be retrieved.
Every merged sense description must be a general reusable meaning, not a description of one specific named object.
When uncertain, merge the candidates and write a broader description.
Return only valid JSON."""


def compact_candidates_for_prompt(candidates: list[dict]) -> list[dict]:
    return [
        {
            "candidate_id": candidate["candidate_id"],
            "label": candidate["label"],
            "hypothesis": candidate["hypothesis"],
        }
        for candidate in candidates
    ]


def compact_fft_samples_for_prompt(sample_records: list[dict]) -> list[dict]:
    rows = []
    for sample in sample_records:
        context_text = re.sub(r"\s+", " ", sample["context_text"]).strip()
        rows.append(
            {
                "sample_order": sample["fft_sample_order"],
                "title": sample["title"],
                "matched_text": sample["matched_text"],
                "context_text": context_text,
            }
        )
    return rows


def build_merge_prompt(word: str, candidates: list[dict], fft_samples: list[dict]) -> str:
    payload = {
        "word": word,
        "candidate_senses": compact_candidates_for_prompt(candidates),
        "fft_dataset_samples": compact_fft_samples_for_prompt(fft_samples),
    }
    return f"""Merge candidate senses for the target word into coarse retrieval-oriented meanings.

Goal:
Create the smallest useful sense inventory for retrieval, using the provided FFT-selected dataset samples as the evidence base. These merged descriptions will later be used by a cross-encoder, so avoid distinctions that are too subtle for short context snippets.

Dataset evidence:
- The fft_dataset_samples are real dataset contexts selected by farthest-first traversal over span embeddings.
- Use all provided samples as the reference for deciding which candidate senses are useful.
- A candidate sense may be included in merged_senses only if at least one provided FFT sample plausibly expresses that sense.
- If a candidate sense does not appear in, or is not supported by, any provided FFT sample, discard it.
- Do not keep a candidate sense merely because it is a valid dictionary or Wikidata sense of the word.

Default bias:
- Merge by broad semantic function, not by Wikidata entity granularity.
- Merge title/domain variants when they are instances of the same role or concept.
- Merge specific subtypes into their broader parent sense unless the subtype changes the entity type or expected evidence.
- If two candidates could both match the same ordinary sentence about the target word, merge them.
- When uncertain, merge.
- A merged sense must describe a general meaning, category, role, function, or concept. Do not write a merged sense as a description of one specific named object, work, organization, place, person, or identifier.
- If the evidence only supports one specific named item and cannot be generalized into a reusable lexical meaning, discard that candidate instead of creating a specific named-object sense.

Split only when:
1. The meanings are genuinely different entity types or concepts, such as fruit vs company or financial bank vs river bank.
2. Keeping them together would make clearly wrong documents look relevant.
3. The distinction is likely obvious from short local context, not just from specialist wording.

Discard only when:
1. The candidate is not a plausible sense of the target word.
2. The candidate is too vague to add value and cannot be merged into a broader valid sense.
3. The candidate is not supported by any of the provided FFT dataset samples.

Output only valid JSON with keys: word, merged_senses, sample_judgments, discarded_candidates, notes.

Each merged_senses item must contain:
- sense_id: a short stable id such as s1, s2, s3
- canonical_label: short label for the merged sense
- merged_description: one sentence describing the broad merged meaning; it must be a general reusable meaning, not a description of one specific named object
- source_candidate_ids: list of integer candidate_id values that were merged
- merge_rationale: one short sentence explaining why these candidates belong together or why the sense stayed separate

Each sample_judgments item must contain:
- sample_order: the integer sample_order from fft_dataset_samples
- matched_text: the matched_text from that sample
- judgment: one of matched_sense, unsupported, ambiguous
- sense_id: the selected merged sense_id when judgment is matched_sense, otherwise null
- confidence: a number from 0.0 to 1.0 indicating confidence in this semantic judgment
- reason: one short sentence explaining the judgment

Each discarded_candidates item must contain:
- candidate_id
- reason

Candidate data:
{json.dumps(payload, ensure_ascii=False, indent=2)}"""


merge_prompt = build_merge_prompt(WORD, candidate_senses, fft_sample_records)
print(merge_prompt)

Merge candidate senses for the target word into coarse retrieval-oriented meanings.

Goal:
Create the smallest useful sense inventory for retrieval, using the provided FFT-selected dataset samples as the evidence base. These merged descriptions will later be used by a cross-encoder, so avoid distinctions that are too subtle for short context snippets.

Dataset evidence:
- The fft_dataset_samples are real dataset contexts selected by farthest-first traversal over span embeddings.
- Use all provided samples as the reference for deciding which candidate senses are useful.
- A candidate sense may be included in merged_senses only if at least one provided FFT sample plausibly expresses that sense.
- If a candidate sense does not appear in, or is not supported by, any provided FFT sample, discard it.
- Do not keep a candidate sense merely because it is a valid dictionary or Wikidata sense of the word.

Default bias:
- Merge by broad semantic function, not by Wikidata entity granularity.
- Me

In [5]:
from openai import OpenAI


def make_openai_client():
    client_kwargs = {}
    if OPENAI_API_KEY:
        client_kwargs["api_key"] = OPENAI_API_KEY
    if OPENAI_BASE_URL:
        client_kwargs["base_url"] = OPENAI_BASE_URL
    return OpenAI(**client_kwargs)


def merge_candidate_senses_with_llm(
    word: str,
    candidates: list[dict],
    fft_samples: list[dict],
    model: str = OPENAI_MODEL,
):
    if not OPENAI_API_KEY:
        raise RuntimeError("OpenAI API key was not found in OPENAI_API_KEY or the root API_KEY file.")

    client = make_openai_client()
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_merge_prompt(word, candidates, fft_samples)},
        ],
    )
    content = response.choices[0].message.content
    return json.loads(content), response


llm_result, raw_response = merge_candidate_senses_with_llm(WORD, candidate_senses, fft_sample_records)
usage = raw_response.usage
token_usage = {
    "prompt_tokens": getattr(usage, "prompt_tokens", None),
    "completion_tokens": getattr(usage, "completion_tokens", None),
    "total_tokens": getattr(usage, "total_tokens", None),
}
print(token_usage)
llm_result

{'prompt_tokens': 3017, 'completion_tokens': 1494, 'total_tokens': 4511}


{'word': 'space',
 'merged_senses': [{'sense_id': 's1',
   'canonical_label': 'physical space / spatial extent',
   'merged_description': 'The three-dimensional extent or region in which objects exist and events occur, including general physical space and outer space as the expanse beyond Earth.',
   'source_candidate_ids': [1, 3, 6],
   'merge_rationale': 'These candidates all refer to broad spatial extent or the physical cosmos, and short contexts often do not require separating universe from outer space or general space.'},
  {'sense_id': 's2',
   'canonical_label': 'blank or open area',
   'merged_description': 'An empty, open, or available area or interval, including whitespace in text and open land or room for use.',
   'source_candidate_ids': [2],
   'merge_rationale': 'The evidence supports a general sense of an empty interval or area, which covers both written spacing and open physical space.'},
  {'sense_id': 's3',
   'canonical_label': 'color space',
   'merged_description':

## 4. Inspect The Merged Candidate Set

In [6]:
merged_df = pd.DataFrame(llm_result.get("merged_senses", []))
merged_df

,sense_id,canonical_label,merged_description,source_candidate_ids,merge_rationale
0,s1,physical space / spatial extent,The three-dimensional extent or region in whic...,"[1, 3, 6]",These candidates all refer to broad spatial ex...
1,s2,blank or open area,"An empty, open, or available area or interval,...",[2],The evidence supports a general sense of an em...
2,s3,color space,A mathematical or technical system for represe...,[5],The FFT samples repeatedly use 'space' in the ...


In [7]:
sample_judgments_df = pd.DataFrame(llm_result.get("sample_judgments", []))
sample_judgments_df

,sample_order,matched_text,judgment,sense_id,confidence,reason
0,1,space,matched_sense,s1,0.98,The context explicitly describes an infinite t...
1,2,space,matched_sense,s2,0.86,The phrase 'in the space of four years' uses s...
2,3,spaces,matched_sense,s3,0.99,"The context is about HCL, RGB, and other color..."
3,4,space,matched_sense,s1,0.93,The title 'Muppets from Space' uses the outer-...
4,5,space,matched_sense,s3,0.99,The passage defines a color space as a color r...
5,6,space,matched_sense,s3,0.99,RGB color space is clearly the technical color...
6,7,space,matched_sense,s3,0.99,The context again refers to color spaces.
7,8,space,matched_sense,s2,0.88,Open space here means undeveloped or open land...
8,9,space,matched_sense,s1,0.91,The phrase 'space race' and landing on Mars in...
9,10,spaces,matched_sense,s2,0.84,The gardens are described as spaces in the sen...


In [8]:
discarded_df = pd.DataFrame(llm_result.get("discarded_candidates", []))
discarded_df

,candidate_id,reason
0,4,The video game sense is not supported by any F...
1,5,The hackspace named entity is not supported by...
2,7,The company name is not supported by any FFT s...
3,8,This is a different word and not a sense of 's...


## 5. Save Experiment Output

In [9]:
# OUTPUT_DIR = REPO_ROOT / "cache" / "wikidata_llm_candidate_merge"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# output_path = OUTPUT_DIR / f"{WORD.replace(' ', '_')}_max{MAX_CANDIDATE_COUNT}_merged.json"
#
# record = {
#     "word": WORD,
#     "max_candidate_count": MAX_CANDIDATE_COUNT,
#     "use_detailed_description": USE_DETAILED_DESCRIPTION,
#     "openai_model": OPENAI_MODEL,
#     "token_usage": token_usage,
#     "raw_candidates": candidate_senses,
#     "fft_dataset_samples": compact_fft_samples_for_prompt(fft_sample_records),
#     "llm_result": llm_result,
# }
# output_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
# print(output_path)